# LLM Benchmarking for Financial Sentiment Analysis
**BANA 275 - Group 16 | University of Cincinnati**

This notebook benchmarks Gemini 1.5 Pro, Llama 3.1 70B and Mixtral 8x7B on the WSB dataset using zero-shot, few-shot, and chain-of-thought prompting.

## Setup
1. Upload `WSB_full.csv` to Colab
2. Set your API keys in the cells below
3. Run all cells

In [ ]:
!pip install openai anthropic google-generativeai together --quiet

In [ ]:
!pip install google-generativeai groq huggingface_hub --quiet

In [ ]:
!pip install groq --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 5.0 MB/s eta 0:00:00


In [ ]:
import os

# GET FREE KEYS (2 min each):
# Gemini: aistudio.google.com/apikey
# Groq:   console.groq.com/keys
# HF:     huggingface.co/settings/tokens (optional)

os.environ['GOOGLE_API_KEY'] = ''
os.environ['GROQ_API_KEY'] = ''
os.environ['HF_API_TOKEN'] = ''

for key, name in [('GOOGLE_API_KEY','Gemini'), ('GROQ_API_KEY','Groq/Llama'), ('HF_API_TOKEN','HuggingFace')]:
    print(f'{name}: {"READY" if os.environ.get(key) else "MISSING"}')

Gemini: READY
Groq/Llama: READY
HuggingFace: READY


In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ['GOOGLE_API_KEY'])
model = genai.GenerativeModel('gemini-2.5-flash')
r = model.generate_content("Say hello")
print(r.text)

Hello!


In [ ]:
import pandas as pd
import numpy as np
import re, time, json
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('WSB_full.csv')
print(f'Total rows: {len(df)}')
print(f'Labels:\n{df["label"].value_counts().sort_index()}')

LABEL_MAP = {0: 0, 1: 1, 2: -1}

# Adjust sample size based on time budget
# 200 = ~10-15 min per model-prompt combo
# 300 = ~20 min per model-prompt combo
SAMPLE_SIZE = 200
eval_df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f'\nEvaluation sample: {len(eval_df)} rows')

Total rows: 2920
Labels:
label
0     930
1    1134
2     856
Name: count, dtype: int64

Evaluation sample: 200 rows


In [ ]:
ZERO_SHOT = '''Classify the following financial social media post into exactly one sentiment category.

Categories:
- Bullish (positive sentiment toward the stock/market)
- Bearish (negative sentiment toward the stock/market)
- Neutral (no clear positive or negative sentiment)

Post: "{text}"

Respond with exactly one word: Bullish, Bearish, or Neutral.'''


FEW_SHOT = '''Classify financial social media posts into sentiment categories.

Examples:
Post: "TSLA to the moon! Just loaded up on more calls, this stock is unstoppable"
Sentiment: Bullish

Post: "Markets are crashing, sold everything. This is going to get much worse"
Sentiment: Bearish

Post: "Anyone know when the next earnings report is? Trying to figure out my timeline"
Sentiment: Neutral

Post: "GME squeeze is happening, diamond hands baby, we are not selling"
Sentiment: Bullish

Post: "Lost 50% of my portfolio this week. Worst decision ever buying those options"
Sentiment: Bearish

Post: "Volume seems normal today, nothing unusual happening with the stock"
Sentiment: Neutral

Now classify:
Post: "{text}"

Respond with exactly one word: Bullish, Bearish, or Neutral.'''


COT = '''You are a financial sentiment analyst. Classify this social media post.

Post: "{text}"

Think step by step:
1. Identify financial terms, tickers, or market references
2. Determine if the author expresses a positive (bullish), negative (bearish), or no clear view (neutral)
3. Consider slang ("moon" = bullish, "diamond hands" = holding, "crash" = bearish)
4. If mixed signals, determine which sentiment dominates

Respond in this format:
Reasoning: [1-2 sentences]
Sentiment: [Bullish/Bearish/Neutral]'''

PROMPTS = {'Zero-shot': ZERO_SHOT, 'Few-shot': FEW_SHOT, 'Chain-of-Thought': COT}
print('Prompts loaded.')

Prompts loaded.


In [ ]:
def call_gemini(text, prompt_tpl, model='gemini-2.5-flash'):
    import google.generativeai as genai
    genai.configure(api_key=os.environ['GOOGLE_API_KEY'])
    m = genai.GenerativeModel(model)
    try:
        r = m.generate_content(prompt_tpl.format(text=text),
            generation_config=genai.GenerationConfig(temperature=0, max_output_tokens=150))
        return r.text.strip()
    except Exception as e:
        return f'ERROR: {e}'

def call_groq(text, prompt_tpl, model='llama-3.1-70b-versatile'):
    from groq import Groq
    client = Groq(api_key=os.environ['GROQ_API_KEY'])
    try:
        r = client.chat.completions.create(model=model,
            messages=[{'role': 'system', 'content': 'You are a financial sentiment classifier.'},
                      {'role': 'user', 'content': prompt_tpl.format(text=text)}],
            temperature=0, max_tokens=150)
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f'ERROR: {e}'

def call_hf(text, prompt_tpl, model='mistralai/Mixtral-8x7B-Instruct-v0.1'):
    from huggingface_hub import InferenceClient
    client = InferenceClient(token=os.environ.get('HF_API_TOKEN', ''))
    try:
        r = client.text_generation(prompt_tpl.format(text=text),
            model=model, max_new_tokens=150, temperature=0.01)
        return r.strip()
    except Exception as e:
        return f'ERROR: {e}'

# ===== CHOOSE MODELS TO RUN =====
# Comment/uncomment as needed. All are free.
MODELS = {
    #'Gemini 2.5 Flash':   {'fn': call_gemini, 'id': 'gemini-2.5-flash',           'delay': 4.5},
    'Gemini 2.5 Flash': {'fn': call_gemini, 'id': 'gemini-2.5-flash',         'delay': 2.5},
    'Llama 3.1 70B':    {'fn': call_groq,   'id': 'llama-3.1-70b-versatile',  'delay': 2.5},
    #'Llama 3.3 70B':   {'fn': call_groq,   'id': 'llama-3.3-70b-versatile',  'delay': 2.5},
    'Mixtral 8x7B':    {'fn': call_hf,     'id': 'mistralai/Mixtral-8x7B-Instruct-v0.1', 'delay': 3.0},
}
print(f'Models to run: {list(MODELS.keys())}')

Models to run: ['Gemini 2.5 Flash', 'Llama 3.1 70B', 'Mixtral 8x7B']


In [ ]:
# def parse_sentiment(raw):
#     t = str(raw).lower().strip()
#     if 'sentiment:' in t:
#         t = t.split('sentiment:')[-1].strip().split('\n')[0]
#     t = re.sub(r'[^a-z]', '', t)

#     if 'bullish' in t:
#         return 1
#     if 'bearish' in t:
#         return -1
#     if 'neutral' in t:
#         return 0
#     return None

# all_results = []

# FAST_DELAY = {
#     'Gemini 2.0 Flash': 0.2,
#     'Llama 3.1 70B': 0.0,
#     'Mixtral 8x7B': 0.3,
# }

# # Keep only the prompts you really need
# PROMPTS = {
#     'Zero-shot': ZERO_SHOT,
#     'Few-shot': FEW_SHOT,
#     # 'CoT': COT,   # leave off unless you really want it
# }

# for model_name, cfg in MODELS.items():
#     for prompt_name, prompt_tpl in PROMPTS.items():
#         print(f'\n{"="*50}')
#         print(f'>>> {model_name} / {prompt_name}')
#         print(f'{"="*50}')

#         preds, errors, retries = [], 0, 0
#         delay = FAST_DELAY.get(model_name, 0.1)

#         for row in tqdm(list(eval_df.itertuples(index=False)), total=len(eval_df)):
#             raw = cfg['fn'](str(row.text)[:1000], prompt_tpl, model=cfg['id'])

#             # retry once if rate limited
#             if 'ERROR' in str(raw) and ('rate' in str(raw).lower() or '429' in str(raw)):
#                 retries += 1
#                 time.sleep(3)
#                 raw = cfg['fn'](str(row.text)[:1000], prompt_tpl, model=cfg['id'])

#             p = parse_sentiment(raw)
#             if p is None:
#                 errors += 1
#                 p = 0

#             preds.append(p)

#             if delay > 0:
#                 time.sleep(delay)

#         true = eval_df['label'].map(LABEL_MAP).values
#         acc = accuracy_score(true, preds)
#         mf1 = f1_score(true, preds, average='macro', zero_division=0)
#         prec, rec, _, _ = precision_recall_fscore_support(
#             true, preds, average='macro', zero_division=0
#         )
#         cm = confusion_matrix(true, preds, labels=[-1, 0, 1])

#         result = {
#             'model': model_name,
#             'prompt': prompt_name,
#             'accuracy': round(acc, 4),
#             'macro_f1': round(mf1, 4),
#             'macro_p': round(prec, 4),
#             'macro_r': round(rec, 4),
#             'cm': cm.tolist(),
#             'errors': errors
#         }
#         all_results.append(result)

#         print(f'Accuracy: {acc:.4f} | Macro F1: {mf1:.4f} | Errors: {errors} | Retries: {retries}')

#         pd.DataFrame(
#             [{k: v for k, v in r.items() if k != 'cm'} for r in all_results]
#         ).to_csv('benchmark_progress.csv', index=False)

def parse_sentiment(raw):
    t = raw.lower().strip()
    if 'sentiment:' in t:
        t = t.split('sentiment:')[-1].strip().split('\n')[0]
    t = re.sub(r'[^a-z]', '', t)
    if 'bullish' in t: return 1
    if 'bearish' in t: return -1
    if 'neutral' in t: return 0
    return None

all_results = []

for model_name, cfg in MODELS.items():
    for prompt_name, prompt_tpl in PROMPTS.items():
        print(f'\n{"="*50}')
        print(f'>>> {model_name} / {prompt_name}')
        print(f'{"="*50}')
        preds, errors = [], 0

        for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
            raw = cfg['fn'](str(row['text'])[:1500], prompt_tpl, model=cfg['id'])
            p = parse_sentiment(raw)
            if p is None:
                errors += 1
                p = 0
            preds.append(p)
            time.sleep(cfg['delay'])

        true = eval_df['label'].map(LABEL_MAP).values
        acc = accuracy_score(true, preds)
        mf1 = f1_score(true, preds, average='macro', zero_division=0)
        prec, rec, _, _ = precision_recall_fscore_support(true, preds, average='macro', zero_division=0)
        cm = confusion_matrix(true, preds, labels=[-1, 0, 1])

        result = {'model': model_name, 'prompt': prompt_name,
                  'accuracy': round(acc, 4), 'macro_f1': round(mf1, 4),
                  'macro_p': round(prec, 4), 'macro_r': round(rec, 4),
                  'cm': cm, 'errors': errors}
        all_results.append(result)
        print(f'  Accuracy: {acc:.4f} | Macro F1: {mf1:.4f} | Parse errors: {errors}')

print(f'\nDone! {len(all_results)} runs completed.')


>>> Gemini 2.5 Flash / Zero-shot


  0%|          | 0/200 [00:00<?, ?it/s]

  Accuracy: 0.3900 | Macro F1: 0.2849 | Parse errors: 179

>>> Gemini 2.5 Flash / Few-shot


  0%|          | 0/200 [00:00<?, ?it/s]

  Accuracy: 0.3500 | Macro F1: 0.1807 | Parse errors: 199

>>> Gemini 2.5 Flash / Chain-of-Thought


  0%|          | 0/200 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Save whatever results exist in all_results
import json

# Save summary CSV
rows = [{k: v for k, v in r.items() if k != 'cm'} for r in all_results]
pd.DataFrame(rows).to_csv('benchmark_progress.csv', index=False)

# Save full results including confusion matrices
save_data = []
for r in all_results:
    entry = {k: v for k, v in r.items()}
    if 'cm' in entry:
        entry['cm'] = entry['cm'].tolist() if hasattr(entry['cm'], 'tolist') else entry['cm']
    save_data.append(entry)

with open('benchmark_full.json', 'w') as f:
    json.dump(save_data, f, indent=2)

print(f'Saved {len(all_results)} runs')
print('Files: benchmark_progress.csv, benchmark_full.json')
pd.DataFrame(rows)

Saved 7 runs
Files: benchmark_progress.csv, benchmark_full.json


,model,prompt,accuracy,macro_f1,macro_p,macro_r,errors
0,Gemini 2.0 Flash,Zero-shot,0.345,0.171,0.115,0.3333,200
1,Gemini 2.0 Flash,Few-shot,0.345,0.171,0.115,0.3333,200
2,Gemini 2.0 Flash,Chain-of-Thought,0.345,0.171,0.115,0.3333,200
3,Llama 3.1 70B,Chain-of-Thought,0.345,0.171,0.115,0.3333,200
4,Llama 3.1 70B,Few-shot,0.345,0.171,0.115,0.3333,200
5,Llama 3.1 70B,Zero-shot,0.345,0.171,0.115,0.3333,200
6,Mixtral 8x7B,Chain-of-Thought,0.345,0.171,0.115,0.3333,200


In [ ]:
# USE IFF COLLAB DISCONNECTS - to continue from benchmark_progress.csv
# ONLY use this if Colab crashed and you lost all_results
# Otherwise skip this cell entirely

# Load previous results and continue from where you left off
import json

# Reload
with open('benchmark_full.json', 'r') as f:
    saved = json.load(f)

all_results = []
for r in saved:
    r['cm'] = np.array(r['cm'])
    all_results.append(r)

# Show what's already done
done = set((r['model'], r['prompt']) for r in all_results)
print(f'Loaded {len(all_results)} completed runs:')
for model, prompt in done:
    print(f'  {model} / {prompt}')

# Figure out what's remaining
all_combos = set((m, p) for m in MODELS for p in PROMPTS)
remaining = all_combos - done
print(f'\nRemaining: {len(remaining)} runs')
for model, prompt in sorted(remaining):
    print(f'  {model} / {prompt}')

# Run only the remaining ones
for model_name, prompt_name in sorted(remaining):
    cfg = MODELS[model_name]
    prompt_tpl = PROMPTS[prompt_name]

    print(f'\n{"="*50}')
    print(f'>>> {model_name} / {prompt_name}')
    print(f'{"="*50}')
    preds, errors = [], 0

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        raw = cfg['fn'](str(row['text'])[:1500], prompt_tpl, model=cfg['id'])
        p = parse_sentiment(raw)
        if p is None:
            errors += 1
            p = 0
        preds.append(p)
        time.sleep(cfg['delay'])

    true = eval_df['label'].map(LABEL_MAP).values
    acc = accuracy_score(true, preds)
    mf1 = f1_score(true, preds, average='macro', zero_division=0)
    prec, rec, _, _ = precision_recall_fscore_support(true, preds, average='macro', zero_division=0)
    cm = confusion_matrix(true, preds, labels=[-1, 0, 1])

    result = {'model': model_name, 'prompt': prompt_name,
              'accuracy': round(acc, 4), 'macro_f1': round(mf1, 4),
              'macro_p': round(prec, 4), 'macro_r': round(rec, 4),
              'cm': cm, 'errors': errors}
    all_results.append(result)
    print(f'  Accuracy: {acc:.4f} | Macro F1: {mf1:.4f} | Parse errors: {errors}')

    # Save after each run
    save_data = []
    for r in all_results:
        entry = {k: v for k, v in r.items()}
        if hasattr(entry.get('cm'), 'tolist'):
            entry['cm'] = entry['cm'].tolist()
        save_data.append(entry)
    with open('benchmark_full.json', 'w') as f:
        json.dump(save_data, f, indent=2)
    print(f'  Saved. Total: {len(all_results)} runs done.')

print(f'\nAll done! {len(all_results)} total runs.')

Loaded 3 completed runs:
  Gemini 2.0 Flash / Chain-of-Thought
  Gemini 2.0 Flash / Zero-shot
  Gemini 2.0 Flash / Few-shot

Remaining: 6 runs
  Llama 3.1 70B / Chain-of-Thought
  Llama 3.1 70B / Few-shot
  Llama 3.1 70B / Zero-shot
  Mixtral 8x7B / Chain-of-Thought
  Mixtral 8x7B / Few-shot
  Mixtral 8x7B / Zero-shot

>>> Llama 3.1 70B / Chain-of-Thought


  0%|          | 0/200 [00:00<?, ?it/s]

  Accuracy: 0.3450 | Macro F1: 0.1710 | Parse errors: 200
  Saved. Total: 4 runs done.

>>> Llama 3.1 70B / Few-shot


  0%|          | 0/200 [00:00<?, ?it/s]

  Accuracy: 0.3450 | Macro F1: 0.1710 | Parse errors: 200
  Saved. Total: 5 runs done.

>>> Llama 3.1 70B / Zero-shot


...

  0%|          | 0/200 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


  Accuracy: 0.3450 | Macro F1: 0.1710 | Parse errors: 200
  Saved. Total: 7 runs done.

>>> Mixtral 8x7B / Few-shot


  0%|          | 0/200 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
models = sorted(set(r['model'] for r in all_results),
                key=lambda m: max(r['macro_f1'] for r in all_results if r['model']==m), reverse=True)
prompts = list(PROMPTS.keys())
mat = np.zeros((len(models), len(prompts)))
for r in all_results:
    mat[models.index(r['model']), prompts.index(r['prompt'])] = r['macro_f1']

fig, ax = plt.subplots(figsize=(8, max(4, len(models)*0.9)))
sns.heatmap(mat, annot=True, fmt='.2f', cmap='RdYlGn',
            xticklabels=prompts, yticklabels=models,
            vmin=0.4, vmax=1.0, linewidths=0.5,
            annot_kws={'size': 14, 'weight': 'bold'}, ax=ax)
ax.set_title('Model x Prompt Strategy: Macro F1 Score\n(WSB + Stock Tweets datasets)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Prompt Strategy'); ax.set_ylabel('Model')
best = np.unravel_index(np.argmax(mat), mat.shape)
ax.annotate('\u2605', xy=(best[1]+0.5, best[0]+0.8), fontsize=16, color='gold', ha='center')
ax.text(0.5, -0.08, '\u2605 Gold star = best model + prompt combination',
        transform=ax.transAxes, fontsize=8, ha='center', color='gray')
plt.tight_layout()
plt.savefig('macro_f1_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Best: {models[best[0]]} / {prompts[best[1]]} = {mat[best]:.2f}')

In [ ]:
sr = sorted(all_results, key=lambda x: x['macro_f1'])
worst, best_r = sr[0], sr[-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Confusion Matrices: Best vs. Worst Model Performance', fontsize=13, fontweight='bold')
labs = ['Bearish', 'Neutral', 'Bullish']

for ax, r, cmap in zip(axes, [best_r, worst], ['Blues', 'Oranges']):
    cm = np.array(r['cm'])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap=cmap,
                xticklabels=labs, yticklabels=labs,
                vmin=0, vmax=1, ax=ax, linewidths=0.5)
    for i in range(3):
        for j in range(3):
            ax.text(j+0.5, i+0.75, f'(n={cm[i][j]})', ha='center', va='center', fontsize=7, color='gray')
    ax.set_title(f"{r['model']} ({r['prompt']})", fontsize=10)
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrices_best_worst.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
rows = [{'Model': r['model'], 'Prompt': r['prompt'],
         'Accuracy': r['accuracy'], 'Macro F1': r['macro_f1'],
         'Macro P': r['macro_p'], 'Macro R': r['macro_r'],
         'Errors': r['errors']} for r in all_results]

summary = pd.DataFrame(rows).sort_values('Macro F1', ascending=False)
summary.to_csv('benchmark_summary.csv', index=False)
print(summary.to_string(index=False))
print('\nSaved: benchmark_summary.csv')

print('\n' + '='*50)
print('COMPARISON WITH TRADITIONAL MODELS')
print('='*50)
print(f'VADER baseline:  Acc 44.18% | Macro F1 0.408')
print(f'FinBERT baseline: Acc 32.95% | Macro F1 0.218')
print(f'Best LLM:        Acc {summary.iloc[0]["Accuracy"]*100:.1f}%  | Macro F1 {summary.iloc[0]["Macro F1"]:.3f}')
print(f'Worst LLM:       Acc {summary.iloc[-1]["Accuracy"]*100:.1f}%  | Macro F1 {summary.iloc[-1]["Macro F1"]:.3f}')